# Example 6 (JAX): A Grassland Fire Simulation

This is the JAX port of `swirl_lm_public_demo6_fire.ipynb`. It exercises:
- Variable density solver with combustion (wood model)
- Smagorinsky SGS model
- Synthetic turbulent inflow
- Immersed boundary method (sponge)
- Rayleigh damping layer
- TKE from turbulent viscosity
- Multi-device execution via `shard_map`

In [ ]:
import collections
import itertools

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from swirl_lm.jax.base import parameters as parameters_lib
from swirl_lm.jax.boundary_condition import immersed_boundary_method
from swirl_lm.jax.boundary_condition import rayleigh_damping_layer
from swirl_lm.jax.boundary_condition import synthetic_turbulent_inflow
from swirl_lm.jax.core import driver as driver_lib
from swirl_lm.jax.physics.combustion import turbulent_kinetic_energy
from swirl_lm.jax.physics.combustion import wood
from swirl_lm.jax.physics.thermodynamics import manager as thermodynamics_manager

# Simulation Setup
To initialize a simulation, one needs to provide the following:
* A text proto file that configures the simulation
* A set of functions that define simulation-specific physics

In [ ]:
# Simulation configuration.
config = parameters_lib.SwirlLMParameters.config_from_text_proto(
    """
    # proto-file: swirl_lm/base/parameters.proto
    # proto-message: SwirlLMParameters

    solver_procedure: VARIABLE_DENSITY
    convection_scheme: CONVECTION_SCHEME_QUICK
    diffusion_scheme: DIFFUSION_SCHEME_CENTRAL_3
    time_integration_scheme: TIME_SCHEME_CN_EXPLICIT_ITERATION
    grid_params {
      # The number of cores in 3 dimensions.
        computation_shape {
          dim_0: 1
          dim_1: 1
          dim_2: 8
        }
      # The physical size of the simulation domain in units of m.
      length {
        dim_0: 200.0
        dim_1: 200.0
        dim_2: 200.0
      }
      # The number of grid points per core in 3 dimensions including ghost cells
      # (halos).
      grid_size {
        dim_0: 128
        dim_1: 128
        dim_2: 20
      }
      # The width of the ghost cells on each side of the domain. It is set to 2
      # considering the stencil width of the QUICK scheme.
      halo_width: 2
      # The time step size in units of s.
      dt: 1e-2
      # The size of the convolution kernel to be used for fundamental numerical
      # operations.
      kernel_size: 16
      periodic {
        dim_0: false dim_1: false dim_2: false
      }
    }
    pre_post_process_info {
      from_config {
        apply_preprocess: true
        preprocess_step_id: 100
        preprocess_periodic: false
        apply_postprocess: false
        postprocess_step_id: 0
        postprocess_periodic: false
      }
    }
    use_sgs: true
    sgs_model {
      smagorinsky {
        c_s: 0.18
        pr_t: 1.0
      }
    }
    gravity_direction {
      dim_0: 0.0 dim_1: 0.0 dim_2: -1.0
    }
    pressure {
      solver {
        jacobi {
          max_iterations: 10 halo_width: 2 omega: 0.67
        }
      }
      num_d_rho_filter: 3
      pressure_outlet: true
    }
    thermodynamics {
      ideal_gas_law {
        const_theta: 283.0
      }
    }
    combustion {
      wood {
        s_b: 0.5
        s_x: 0.05
        h_conv: 50.0
        a_v: 8.0
        cp_g: 1200.0
        h_f: 1.96e7
        t_pyr: 600.0
        n_step: 1
        include_radiation: true
        t_ambient: 283.0
        efficiency: 1.0
        c_f: 0.5
        moist_wood {
          h_w: 2.2564e6
          t_vap: 373.0
          c_w: 1.0
        }
        tke {
          turbulent_viscosity {}
        }
      }
    }
    density: 1.2228709548
    p_thermal: 1.01325e5
    kinematic_viscosity: 1.0e-5
    scalars {
      name: "theta"
      diffusivity: 1.0e-5
      reference_value: 283.0
    }
    scalars {
      name: "Y_O"
      diffusivity: 1.0e-5
      density: 1.2228709548
      molecular_weight: 0.029
    }
    scalars {
      name: "ambient"
      diffusivity: 1.0e-5
      density: 1.2228709548
      molecular_weight: 0.029
      solve_scalar: false
    }
    boundary_models {
      ib {
        sponge {
          damping_coeff: 20.0
          variables { name: "u" value: 0.0 bc: DIRICHLET override: false }
          variables { name: "v" value: 0.0 bc: DIRICHLET override: false }
          variables { name: "w" value: 0.0 bc: DIRICHLET override: false }
          variables { name: "theta" value: 283.0 bc: DIRICHLET override: false }
          variables { name: "Y_O" value: 0.21 bc: DIRICHLET override: false }
        }
      }
      most {
        z_0: 0.15
        beta_m: 4.8
        beta_h: 7.8
        alpha: 1.0
        t_s: 283.0
        enable_theta_reg: true
        theta_max: 283.0
        theta_min: 283.0
        active_scalar: "theta"
        sea_level_ref {
          name: "theta"
          value: 283.0
        }
      }
      sponge {
        orientation {dim: 0 face: 1 fraction: 0.1}
        orientation {dim: 2 face: 1 fraction: 0.1}
        variable_info {
          name: "u" target_state_name: "u_init"
        }
        variable_info {
          name: "v" target_state_name: "v_init"
        }
        variable_info {
          name: "w" target_state_name: "w_init"
        }
        variable_info {
          name: "theta" target_value: 283.0
        }
        variable_info {
          name: "Y_O" target_value: 0.21
        }
      }
    }
    additional_state_keys: "rho_f"
    additional_state_keys: "rho_f_init"
    additional_state_keys: "rho_m"
    additional_state_keys: "phi_w"
    additional_state_keys: "T_s"
    additional_state_keys: "src_u"
    additional_state_keys: "src_v"
    additional_state_keys: "src_w"
    additional_state_keys: "src_rho"
    additional_state_keys: "src_theta"
    additional_state_keys: "src_Y_O"
    additional_state_keys: "nu_t"
    additional_state_keys: "tke"
    additional_state_keys: "ignition_kernel"
    additional_state_keys: "bc_u_0_0"
    additional_state_keys: "bc_v_0_0"
    additional_state_keys: "bc_w_0_0"
    additional_state_keys: "ib_interior_mask"
    additional_state_keys: "ib_boundary"
    additional_state_keys: "sponge_beta"
    additional_state_keys: "u_init"
    additional_state_keys: "v_init"
    additional_state_keys: "w_init"
    additional_state_keys: "zz"
    helper_var_keys: "mean_u_0_0"
    helper_var_keys: "mean_v_0_0"
    helper_var_keys: "mean_w_0_0"
    helper_var_keys: "rms_u_0_0"
    helper_var_keys: "rms_v_0_0"
    helper_var_keys: "rms_w_0_0"
    helper_var_keys: "rand_u_0_0"
    helper_var_keys: "rand_v_0_0"
    helper_var_keys: "rand_w_0_0"

    boundary_conditions {
      name: "u"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_DIRICHLET value: 8.5
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    boundary_conditions {
      name: "v"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_DIRICHLET value: 0.0
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_DIRICHLET value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_DIRICHLET value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    boundary_conditions {
      name: "w"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_DIRICHLET value: 0.0
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_DIRICHLET value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    boundary_conditions {
      name: "p"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    boundary_conditions {
      name: "theta"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_DIRICHLET value: 283.0
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    boundary_conditions {
      name: "Y_O"
      boundary_info {
        dim: 0 location: 0 type: BC_TYPE_DIRICHLET value: 0.21
      }
      boundary_info {
        dim: 0 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 1 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 0 type: BC_TYPE_NEUMANN value: 0.0
      }
      boundary_info {
        dim: 2 location: 1 type: BC_TYPE_NEUMANN value: 0.0
      }
    }
    """,
)

gp = config.grid_params
print(f"Grid: nx={gp.nx}, ny={gp.ny}, nz={gp.nz}")
print(f"Core: core_nx={gp.core_nx}, core_ny={gp.core_ny}, core_nz={gp.core_nz}")
print(f"Topology: cx={gp.cx}, cy={gp.cy}, cz={gp.cz}")
print(f"Devices available: {jax.device_count()}")
print(
    f"Preprocess: apply={config.apply_preprocess},"
    f" step_id={config.preprocess_step_id}"
)

In [ ]:
# Defines simulation specific parameters and models.

# Fire related parameters.
fuel_bed_height = 4.0
fuel_load = 0.6
moisture_content = 0.142

fuel_density = fuel_load / 1.5
moisture_density = moisture_content * fuel_density

c_d = 0.01
a_v = 8.0

# Ignition parameters.
SlantLine = collections.namedtuple(
    'SlantLine', ['center_x', 'center_y', 'length', 'thickness', 'angle']
)
ignition_line_info = SlantLine(
    center_x=20.0, center_y=100.0, length=60.0, thickness=8.0, angle=90.0
)
ignition_temperature = 600.0

# Flow field parameters.
u_rms = 2.0
v_rms = 2.0
w_rms = 2.0

u_init = config.bc['u'][0][0][1]
v_init = config.bc['v'][0][0][1]
t_init = config.bc['theta'][0][0][1]
y_o_init = config.bc['Y_O'][0][0][1]
z_0 = 0.15

# Thermodynamics.
thermodynamics = thermodynamics_manager.ThermodynamicsManager(config)

# Combustion.
combustion = wood.wood_combustion_factory(config)

# Inflow generator.
length_scales = (10.0 * gp.dx, 10.0 * gp.dy, 10.0 * gp.dz)
delta = (gp.dx, gp.dy, gp.dz)
mesh_size = (gp.core_nx, gp.core_ny, gp.core_nz)
inflow_dim = 0
inflow_face = 0
inflow_generator = synthetic_turbulent_inflow.SyntheticTurbulentInflow(
    length_scales, delta, mesh_size, inflow_dim, inflow_face
)

# The Rayleigh damping layer.
sponge = rayleigh_damping_layer.RayleighDampingLayer(config.sponge)

# The immersed boundary method for the terrain representation.
ib = immersed_boundary_method.ImmersedBoundaryMethod(config)

In [ ]:
# Helper function to create per-device meshgrid and initialize a field.
# This replaces TF's `initializer.partial_mesh_for_core`.


def _partial_mesh_for_core(
    params,
    logical_coordinates,
    value_fn,
    pad_mode='SYMMETRIC',
):
  """Creates a 3D field on one device by evaluating `value_fn` on a meshgrid.

  The meshgrid covers the core region of the device (without halos). The
  result is padded with `pad_mode` to include halo layers.

  Args:
    params: Simulation parameters.
    logical_coordinates: (cx_i, cy_i, cz_i) position of this device.
    value_fn: Function(xx, yy, zz, lx, ly, lz) -> 3D array on the core.
    pad_mode: Padding mode for halos. 'CONSTANT': Pad with zeros. 'SYMMETRIC':
      Mirror-reflect at edges. 'REFLECT': Reflect without repeating edge.
      'PHYSICAL': Use physically correct coordinate values in halos from
      `global_xyz_with_halos`. This is essential for coordinate fields like `zz`
      where halo values must represent real physical positions (e.g. for
      hydrostatic pressure computation at device boundaries).

  Returns:
    A 3D array of shape matching data_axis_order with halos.
  """
  gp = params.grid_params
  hw = gp.halo_width

  # Core dimensions per device.
  core_nx = gp.core_nx
  core_ny = gp.core_ny
  core_nz = gp.core_nz

  # Global lengths.
  lx = gp.lx
  ly = gp.ly
  lz = gp.lz

  # This device's position in the computational grid.
  gx, gy, gz = logical_coordinates

  if pad_mode is not None and pad_mode.upper() == 'PHYSICAL':
    # Use global coordinate arrays that include physically correct halo values.
    # global_xyz_with_halos is in data_axis_order, so we need to map
    # device coordinates accordingly.
    core_ns = gp.to_data_axis_order(core_nx, core_ny, core_nz)
    ns = gp.to_data_axis_order(gp.nx, gp.ny, gp.nz)
    device_ids = gp.to_data_axis_order(gx, gy, gz)

    slices_1d = []
    for dim in range(3):
      coord = gp.global_xyz_with_halos[dim]
      start = device_ids[dim] * core_ns[dim]
      end = start + ns[dim]
      slices_1d.append(jnp.array(coord[start:end]))

    # Build meshgrid in data_axis_order directly.
    grids = jnp.meshgrid(*slices_1d, indexing='ij')

    # Convert data_axis_order grids back to xyz for value_fn.
    xyz_grids = gp.to_xyz_order(grids)
    xx, yy, zz = xyz_grids

    result = value_fn(xx, yy, zz, lx, ly, lz)
    # Result is already in data_axis_order (from meshgrid in that order).
    return result

  # Build 1D coordinate arrays for the core region of this device.
  dx = lx / (gp.cx * core_nx)
  dy = ly / (gp.cy * core_ny)
  dz = lz / (gp.cz * core_nz)

  x_start = gx * core_nx * dx
  y_start = gy * core_ny * dy
  z_start = gz * core_nz * dz

  x_1d = jnp.linspace(x_start, x_start + (core_nx - 1) * dx, core_nx)
  y_1d = jnp.linspace(y_start, y_start + (core_ny - 1) * dy, core_ny)
  z_1d = jnp.linspace(z_start, z_start + (core_nz - 1) * dz, core_nz)

  xx, yy, zz = jnp.meshgrid(x_1d, y_1d, z_1d, indexing='ij')

  # Evaluate the value function on the core mesh.
  core_values = value_fn(xx, yy, zz, lx, ly, lz)

  # Pad with halos.
  if pad_mode is None or hw == 0:
    padded = core_values
  elif pad_mode.upper() == 'CONSTANT':
    padded = jnp.pad(core_values, hw, mode='constant', constant_values=0.0)
  elif pad_mode.upper() == 'SYMMETRIC':
    padded = jnp.pad(core_values, hw, mode='symmetric')
  elif pad_mode.upper() == 'REFLECT':
    padded = jnp.pad(core_values, hw, mode='reflect')
  else:
    raise ValueError(f'Unknown pad_mode: {pad_mode}')

  # Transpose from xyz indexing to data_axis_order.
  # meshgrid with indexing='ij' produces (x, y, z) order.
  # We need to permute to data_axis_order.
  perm = gp.get_axis_index(('x', 'y', 'z'))
  padded = jnp.transpose(padded, perm)

  return padded


def _constant_field(params, logical_coordinates, value):
  """Creates a constant 3D field on one device."""
  return _partial_mesh_for_core(
      params,
      logical_coordinates,
      lambda xx, yy, zz, lx, ly, lz: value * jnp.ones_like(xx),
  )


def _logarithmic_boundary_layer_init(
    params, logical_coordinates, u_inf, v_inf, z_0_val
):
  """Initializes velocity with a logarithmic boundary layer profile."""

  def u_fn(xx, yy, zz, lx, ly, lz):
    z_safe = jnp.maximum(zz, z_0_val)
    return u_inf * jnp.log(z_safe / z_0_val) / jnp.log(lz / z_0_val)

  def v_fn(xx, yy, zz, lx, ly, lz):
    z_safe = jnp.maximum(zz, z_0_val)
    return v_inf * jnp.log(z_safe / z_0_val) / jnp.log(lz / z_0_val)

  def w_fn(xx, yy, zz, lx, ly, lz):
    return jnp.zeros_like(xx)

  return {
      'u': _partial_mesh_for_core(params, logical_coordinates, u_fn),
      'v': _partial_mesh_for_core(params, logical_coordinates, v_fn),
      'w': _partial_mesh_for_core(params, logical_coordinates, w_fn),
  }

In [ ]:
# Defines the function that initializes state variables.


def _get_terrain_init_fn(terrain_type):
  """Generates functions that initialize IB to represent the terrain."""

  def gaussian(xx, yy, zz, lx, ly, lz):
    """Generates a bump with the Gaussian profile."""
    del yy, zz, ly

    h_ratio = 0.1
    w_ratio = 0.1

    return (
        h_ratio
        * lz
        * jnp.exp(-((xx - 0.5 * lx) ** 2) / (2.0 * (w_ratio * lx) ** 2))
    )

  if terrain_type == 'gaussian':
    terrain_height_fn = gaussian
  else:
    raise ValueError(
        f'Unrecognized terrain type {terrain_type}. Available options: '
        '"gaussian".'
    )

  def flow_field_mask_fn(xx, yy, zz, lx, ly, lz):
    """Sets values to 1 in the flow field and 0 in the solid."""
    h = terrain_height_fn(xx, yy, zz, lx, ly, lz)
    return jnp.where(zz > h, jnp.ones_like(zz), jnp.zeros_like(zz))

  def ib_boundary_mask_fn(xx, yy, zz, lx, ly, lz):
    """Sets values to 1 at the fluid-solid interface and 0 elsewhere."""
    h = terrain_height_fn(xx, yy, zz, lx, ly, lz)
    dz_val = lz / (gp.cz * gp.core_nz)
    return jnp.where(
        jnp.logical_and(zz <= h, zz > h - dz_val),
        jnp.ones_like(zz),
        jnp.zeros_like(zz),
    )

  return terrain_height_fn, flow_field_mask_fn, ib_boundary_mask_fn


def grassland_fire_init_fn(params, logical_coordinates):
  """Initializes state variables in a grassland fire.

  JAX `InitFn` signature: (params, logical_coordinates) -> dict.
  """
  gp_local = params.grid_params
  hw = gp_local.halo_width

  # Axis indices for data_axis_order-aware slicing.
  ax_x = gp_local.get_axis_index('x')
  ax_y = gp_local.get_axis_index('y')
  ax_z = gp_local.get_axis_index('z')

  def pmfc(value_fn, pad_mode='SYMMETRIC'):
    return _partial_mesh_for_core(
        params, logical_coordinates, value_fn, pad_mode
    )

  # Initializes the terrain if IB is used.
  if ib is not None:
    terrain_height_fn, flow_field_mask_fn, ib_boundary_mask_fn = (
        _get_terrain_init_fn('gaussian')
    )
  else:
    terrain_height_fn = lambda xx, yy, zz, lx, ly, lz: jnp.zeros_like(zz)

  def rho_f_init_fn(xx, yy, zz, lx, ly, lz):
    """Generates initial `rho_f` field."""
    h = terrain_height_fn(xx, yy, zz, lx, ly, lz)
    return jnp.where(
        jnp.logical_and(zz <= h + fuel_bed_height, zz >= h),
        fuel_density * jnp.ones_like(zz),
        jnp.zeros_like(zz),
    )

  def rho_m_init_fn(xx, yy, zz, lx, ly, lz):
    """Generates initial moisture `rho_m` field."""
    h = terrain_height_fn(xx, yy, zz, lx, ly, lz)
    return jnp.where(
        jnp.logical_and(zz <= h + fuel_bed_height, zz >= h),
        moisture_density * jnp.ones_like(zz),
        jnp.zeros_like(zz),
    )

  def ignition_kernel_init_fn(xx, yy, zz, lx, ly, lz):
    """Generates a line ignition kernel."""
    h = terrain_height_fn(xx, yy, zz, lx, ly, lz)

    rad = np.radians(ignition_line_info.angle)
    cos = np.cos(rad)
    sin = np.sin(rad)

    x_new = cos * (xx - ignition_line_info.center_x) + sin * (
        yy - ignition_line_info.center_y
    )
    y_new = -sin * (xx - ignition_line_info.center_x) + cos * (
        yy - ignition_line_info.center_y
    )

    location_x = jnp.logical_and(
        x_new <= ignition_line_info.length / 2.0,
        x_new >= -ignition_line_info.length / 2.0,
    )
    location_y = jnp.logical_and(
        y_new <= ignition_line_info.thickness / 2.0,
        y_new >= -ignition_line_info.thickness / 2.0,
    )
    location_xy = jnp.logical_and(location_x, location_y)

    location_z = jnp.logical_and(zz <= h + fuel_bed_height, zz >= h)
    location = jnp.logical_and(location_xy, location_z)

    return jnp.where(location, jnp.ones_like(zz), jnp.zeros_like(zz))

  # Initialize velocity with logarithmic boundary layer.
  uvw = _logarithmic_boundary_layer_init(
      params, logical_coordinates, u_init, v_init, z_0
  )

  output = {}

  # Initializes the terrain if IB is used.
  if ib is not None:
    # In JAX, IB.generate_initial_states takes (grid_shape, mask, boundary).
    ib_interior_mask = pmfc(flow_field_mask_fn)
    ib_boundary_mask = pmfc(ib_boundary_mask_fn)
    grid_shape = ib_interior_mask.shape
    output.update(
        ib.generate_initial_states(
            grid_shape, ib_interior_mask, ib_boundary_mask
        )
    )

  # Initializes state variables.
  output.update({
      'rho': _constant_field(params, logical_coordinates, config.rho),
      'u': uvw['u'],
      'v': uvw['v'],
      'w': uvw['w'],
      'p': _constant_field(params, logical_coordinates, 0.0),
      'theta': _constant_field(params, logical_coordinates, t_init),
      'Y_O': _constant_field(params, logical_coordinates, y_o_init),
      # Turbulent viscosity for the SGS model.
      'nu_t': _constant_field(params, logical_coordinates, 0.0),
      # Variables required in the fire model.
      'rho_f': pmfc(rho_f_init_fn, 'CONSTANT'),
      'rho_f_init': pmfc(rho_f_init_fn, 'CONSTANT'),
      'T_s': _constant_field(params, logical_coordinates, t_init),
      'src_rho': _constant_field(params, logical_coordinates, 0.0),
      'src_theta': _constant_field(params, logical_coordinates, 0.0),
      'src_Y_O': _constant_field(params, logical_coordinates, 0.0),
      'tke': _constant_field(params, logical_coordinates, 0.0),
      'ignition_kernel': pmfc(ignition_kernel_init_fn),
      'rho_m': pmfc(rho_m_init_fn, 'CONSTANT'),
      'phi_w': _constant_field(params, logical_coordinates, 0.0),
      # Geopotential height. Uses 'PHYSICAL' pad mode so that halo values
      # contain correct physical z-coordinates (essential for hydrostatic
      # pressure computation at device boundaries).
      'zz': pmfc(lambda xx, yy, zz, lx, ly, lz: zz, 'PHYSICAL'),
  })

  if 'ib_interior_mask' in output:
    output.update({
        vel: output[vel] * output['ib_interior_mask'] for vel in ('u', 'v', 'w')
    })

  # Generates variables required by the synthetic turbulent inflow.
  rng_key = jax.random.PRNGKey(42)
  r = inflow_generator.generate_random_fields(rng_key)

  core_nz = gp_local.core_nz
  core_ny = gp_local.core_ny

  def cut_yz_plane(f):
    """Retrieves the y-z plane at the inflow face from `f` without halos.

    The inflow generator expects 2D planes of shape (m[0], m[1]) where
    m = mesh_size[inflow_plane] = (core_ny, core_nz) for inflow_dim=0.
    With data_axis_order=('x','y','z'), squeezing the x-dim from a 3D
    field gives (core_ny, core_nz) directly — no transpose needed.
    TF's cut_yz_plane transposed because data_axis_order=('z','x','y')
    produced (core_nz, core_ny) after squeezing x.
    """
    # Extract a y-z plane at the first core x-point, skipping halos.
    start = [0, 0, 0]
    start[ax_x] = hw
    start[ax_y] = hw
    start[ax_z] = hw
    sizes = list(f.shape)
    sizes[ax_x] = 1
    sizes[ax_y] = core_ny
    sizes[ax_z] = core_nz
    plane = jnp.squeeze(jax.lax.dynamic_slice(f, start, sizes))
    # No transpose: with data_axis_order=('x','y','z'), squeezing x gives
    # (core_ny, core_nz) which matches the inflow generator's (m[0], m[1]).
    return plane

  mask = cut_yz_plane(jnp.ones_like(output['u']))

  # Slice along the x-axis (inflow_dim=0) to get the boundary data.
  # bc_u_0_0 stores the first hw+1 x-planes of the velocity field.
  # Use data_axis_order-aware slicing.
  def _slice_inflow_bc(f):
    """Slices the first hw+1 planes along the x-axis."""
    slices = [slice(None)] * 3
    slices[ax_x] = slice(None, hw + 1)
    return f[tuple(slices)]

  output.update({
      'bc_u_0_0': _slice_inflow_bc(output['u']),
      'bc_v_0_0': _slice_inflow_bc(output['v']),
      'bc_w_0_0': _slice_inflow_bc(output['w']),
      'mean_u_0_0': cut_yz_plane(output['u']),
      'mean_v_0_0': cut_yz_plane(output['v']),
      'mean_w_0_0': cut_yz_plane(output['w']),
      'rms_u_0_0': u_rms * mask,
      'rms_v_0_0': v_rms * mask,
      'rms_w_0_0': w_rms * mask,
      'rand_u_0_0': r[0],
      'rand_v_0_0': r[1],
      'rand_w_0_0': r[2],
  })

  # Generates variables required by the Rayleigh damping layer.
  sponge_orientations = list(next(iter(config.sponge)).orientation)
  sponge_init_fn = rayleigh_damping_layer.klemp_lilly_relaxation_coeff_fn(
      sponge_orientations,
      x0=0.0,
      y0=0.0,
      z0=0.0,
  )
  output.update({'sponge_beta': pmfc(sponge_init_fn)})

  for variable in next(iter(config.sponge)).variable_info:
    # Add the target state if requested.
    if variable.WhichOneof('target') == 'target_state_name':
      target_name = '{}_init'.format(variable.name)
      output.update({target_name: output[variable.name]})

    # The source term from Rayleigh damping for the velocity will be computed
    # by an externally defined source term function.
    if variable.name in ('u', 'v', 'w'):
      continue

    output.update({
        'src_{}'.format(variable.name): _constant_field(
            params, logical_coordinates, 0.0
        ),
    })

  # Source terms for velocity (initialized to zero).
  for vel in ('u', 'v', 'w'):
    src_key = f'src_{vel}'
    if src_key not in output:
      output[src_key] = _constant_field(params, logical_coordinates, 0.0)

  return output

In [ ]:
# Defines a function that ignites the fire at the beginning of the simulation.


def _ignite(ignition_kernel, temperature):
  """Sets a high temperature region at the specified location."""
  return temperature + (ignition_temperature - temperature) * ignition_kernel


def pre_simulation_update_fn(states, additional_states):
  """Ignites the fire.

  JAX signature: (states=, additional_states=) -> dict.
  This function is invoked only once at the step specified by
  `preprocess_step_id` (at the beginning of the simulation in this demo).
  """
  output = dict(states)
  output.update(additional_states)

  temperature = _ignite(additional_states['ignition_kernel'], states['theta'])
  t_s_ign = _ignite(
      additional_states['ignition_kernel'], additional_states['T_s']
  )

  output.update({
      'theta': jnp.where(
          additional_states['rho_f'] > 0.0, temperature, states['theta']
      ),
      'T_s': jnp.where(
          additional_states['rho_f'] > 0.0, t_s_ign, additional_states['T_s']
      ),
  })

  thermo_states = {'theta': output['theta'], 'Y_O': states['Y_O']}
  thermo_additional_states = {}
  rho = thermodynamics.update_thermal_density(
      thermo_states, thermo_additional_states
  )
  output.update({'rho': rho})

  # Rescale velocity so that momentum stays the same as before ignition.
  output.update(
      {vel: states['rho'] * states[vel] / rho for vel in ('u', 'v', 'w')}
  )

  return output

In [ ]:
# Defines a function that updates helper state variables.


def additional_states_update_fn(states, additional_states, step_id):
  """Updates `additional_states` for drag forces and fire physics.

  JAX signature: (states=, additional_states=, step_id=) -> dict.
  """
  del step_id

  states_updated = dict(states)
  additional_states_updated = dict(additional_states)

  # Update the turbulent kinetic energy for the combustion source term
  # computation.
  tke_update_fn = turbulent_kinetic_energy.tke_update_fn_manager(
      config.combustion.wood.tke
  )
  additional_states_updated.update(
      tke_update_fn(
          states=states_updated,
          additional_states=additional_states_updated,
          dx=gp.dx,
          dy=gp.dy,
          dz=gp.dz,
          halo_width=gp.halo_width,
      )
  )

  # Update solid states and reaction source terms for the potential temperature
  # and oxygen mass fraction.
  additional_states_updated.update(
      combustion.dry_wood_update_fn(
          rho_f_init=additional_states.get('rho_f_init')
      )(
          replica_id=jnp.int32(0),
          replicas=np.array([[[0]]]),
          states=states_updated,
          additional_states=additional_states_updated,
      )
  )

  # Updates the inflow.
  inflow_update_fn = inflow_generator.generate_inflow_update_fn()
  additional_states_updated.update(
      inflow_update_fn(
          states=states_updated,
          additional_states=additional_states_updated,
          params=config,
      )
  )

  # Updates the forcing term from the immersed boundary method.
  ib_vars = ('u', 'v', 'w', 'theta', 'Y_O')
  ib_additional_states = {
      f'src_{var}': jnp.zeros_like(states[var]) for var in ib_vars
  }
  ib_helper_vars = ('ib_interior_mask', 'ib_boundary')
  ib_additional_states.update({
      ib_var: additional_states[ib_var]
      for ib_var in ib_helper_vars
      if ib_var in additional_states
  })
  ib_additional_states = ib.update_additional_states(
      states=states,
      additional_states=ib_additional_states,
  )
  for var in ib_vars:
    src_var = f'src_{var}'
    additional_states_updated[src_var] = (
        additional_states_updated[src_var] + ib_additional_states[src_var]
    )

  # Updates the Rayleigh damping forcing terms.
  sponge_states = {
      'Y_O': states_updated['Y_O'],
      'theta': states_updated['theta'],
  }
  sponge_additional_states = {
      'sponge_beta': additional_states['sponge_beta'],
      'src_Y_O': jnp.zeros_like(states_updated['Y_O']),
      'src_theta': jnp.zeros_like(states_updated['theta']),
  }
  for varname in ('theta', 'Y_O'):
    target_name = '{}_init'.format(varname)
    if target_name in additional_states:
      sponge_additional_states[target_name] = additional_states[target_name]

  sponge_additional_states = sponge.additional_states_update_fn(
      states=sponge_states,
      additional_states=sponge_additional_states,
      dt=config.dt,
  )
  for varname in ['src_Y_O', 'src_theta']:
    additional_states_updated[varname] = (
        additional_states_updated[varname] + sponge_additional_states[varname]
    )

  return additional_states_updated

In [ ]:
# Defines a function that computes the source terms for the velocity.

# The threshold below which the fuel is treated as depleted.
EPSILON = 1e-6

# Names of the velocity components.
VELOCITY = ('u', 'v', 'w')


def vegetation_drag_update_fn(c_d, a_v):
  """Generates an update function for drag forces due to vegetation."""

  def update_fn(states, additional_states):
    """Updates 'src_u', 'src_v', 'src_w'."""
    u_mag = jnp.sqrt(states['u'] ** 2 + states['v'] ** 2 + states['w'] ** 2)

    def drag_force(u):
      """Computes the drag force for the specified velocity."""
      drag = -3.0 * c_d * states['rho'] * a_v * u_mag * u / 8.0
      return jnp.where(
          additional_states['rho_f'] < EPSILON,
          jnp.zeros_like(drag),
          drag,
      )

    updated_additional_states = dict(additional_states)
    for varname in VELOCITY:
      updated_additional_states[f'src_{varname}'] = drag_force(states[varname])

    return updated_additional_states

  return update_fn


def source_term_update_fn():
  """Generates a library of functions that updates requested source terms.

  JAX signature for each fn: (states, additional_states) -> dict with src_*.
  """

  def src_uvw_fn(states, additional_states):
    """Updates the source terms for u, v, and w."""
    src = {f'src_{var}': jnp.zeros_like(states[var]) for var in VELOCITY}

    # Canopy drag force.
    helper_states = {
        'rho_f': additional_states['rho_f'],
        'src_u': None,
        'src_v': None,
        'src_w': None,
    }
    drag = vegetation_drag_update_fn(c_d, a_v)(states, helper_states)
    src.update({
        src_name: src[src_name] + drag[src_name]
        for src_name in [f'src_{var}' for var in VELOCITY]
    })

    # Update the Rayleigh damping force.
    sponge_states = {key: states[key] for key in VELOCITY}
    sponge_states['rho'] = states['rho']
    helper_states = {'sponge_beta': additional_states['sponge_beta']}
    helper_states.update(
        {f'src_{var}': jnp.zeros_like(states[var]) for var in VELOCITY}
    )
    helper_states.update({
        target_name: additional_states[target_name]
        for target_name in [f'{var}_init' for var in VELOCITY]
        if target_name in additional_states
    })
    sponge_force = sponge.additional_states_update_fn(
        states=sponge_states,
        additional_states=helper_states,
        dt=config.dt,
    )
    src.update({
        src_name: src[src_name] + states['rho'] * sponge_force[src_name]
        for src_name in [f'src_{var}' for var in VELOCITY]
    })

    return src

  def src_u_fn(states, additional_states):
    """Updates the source term for u."""
    src_uvw = src_uvw_fn(states, additional_states)
    return {'src_u': src_uvw['src_u']}

  def src_v_fn(states, additional_states):
    """Updates the source term for v."""
    src_uvw = src_uvw_fn(states, additional_states)
    return {'src_v': src_uvw['src_v']}

  def src_w_fn(states, additional_states):
    """Updates the source term for w."""
    src_uvw = src_uvw_fn(states, additional_states)
    return {'src_w': src_uvw['src_w']}

  return {'u': src_u_fn, 'v': src_v_fn, 'w': src_w_fn}

In [ ]:
# Updates the simulation parameter handler with the simulation specific
# functions.

config.source_update_fn_lib = source_term_update_fn()
config.additional_states_update_fn = additional_states_update_fn
config.preprocessing_states_update_fn = pre_simulation_update_fn

# Run the simulation

In [ ]:
# Run the simulation using the JAX driver.
num_steps = 200
num_steps_per_cycle = 200

state = driver_lib.run_simulation(
    params=config,
    init_fn=grassland_fire_init_fn,
    num_steps=num_steps,
    num_steps_per_cycle=num_steps_per_cycle,
)

print(
    "Simulation completed. Final time:"
    f" {float(state[driver_lib.TIME_VARNAME]):.4f} s"
)
print(f"State keys: {sorted(state.keys())}")

# Post-processing

In [ ]:
# Results visualization


def _remove_halos(field, gp):
  hw = gp.halo_width

  buf = np.zeros((gp.fx, gp.fy, gp.fz), dtype=field.dtype)
  for i, j, k in itertools.product(range(gp.cx), range(gp.cy), range(gp.cz)):
    buf[
        i * gp.core_nx : (i + 1) * gp.core_nx,
        j * gp.core_ny : (j + 1) * gp.core_ny,
        k * gp.core_nz : (k + 1) * gp.core_nz,
    ] = field[
        i * gp.nx + hw : (i + 1) * gp.nx - hw,
        j * gp.ny + hw : (j + 1) * gp.ny - hw,
        k * gp.nz + hw : (k + 1) * gp.nz - hw,
    ]
  return buf


# For multi-device, the global array is sharded. Gather to host for plotting.
varname = 'w'  # @param['u', 'v', 'w', 'p', 'rho', 'theta', 'Y_O', 'rho_f', 'T_s']

field = _remove_halos(np.asarray(state[varname]), gp)

# field shape is (nx_global, y_global, nz_global) in data_axis_order.
z_slice = 4  # @param{type: 'number'}
y_slice = 64  # @param{type: 'number'}

data_slice_z = field[..., z_slice]  # slice in z
data_slice_y = field[:, y_slice, :]  # slice in y

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
im = ax[0].imshow(
    data_slice_z.T,
    origin='lower',
    cmap='hot',
    aspect='equal',
)
ax[0].set_xlabel('x')
ax[0].set_ylabel('y')
ax[0].set_title(f'{varname} at z-slice {z_slice}')
plt.colorbar(im, ax=ax[0])

im = ax[1].imshow(
    data_slice_y.T,
    origin='lower',
    cmap='hot',
    aspect='equal',
)
ax[1].set_xlabel('x')
ax[1].set_ylabel('z')
ax[1].set_title(f'{varname} at y-slice {y_slice}')
plt.colorbar(im, ax=ax[1])

plt.tight_layout()
plt.show()